In [1]:
def is_prime(n):
    """Check if a number is prime using trial division."""
    if n < 2:
        return False
    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0:
            return False
    return True

def is_blum_prime(p):
    """Check if a prime number is a Blum prime (congruent to 3 mod 4)."""
    return is_prime(p) and p % 4 == 3

def gcd(a, b):
    """Calculate the Greatest Common Divisor of a and b using Euclidean algorithm."""
    while b:
        a, b = b, a % b
    return a

class BBS:
    def __init__(self, p, q, seed):
        """Initialize BBS PRNG with two Blum primes and a seed.
        
        Args:
            p (int): First Blum prime
            q (int): Second Blum prime
            seed (int): Seed value, must be coprime to n = p*q
        """
        if not (is_blum_prime(p) and is_blum_prime(q)):
            raise ValueError("p and q must both be Blum primes (prime numbers congruent to 3 mod 4)")
        
        self.n = p * q
        
        if gcd(seed, self.n) != 1:
            raise ValueError("Seed must be coprime to n")
        
        self.state = (seed * seed) % self.n

    def next_bit(self):
        """Generate the next bit in the sequence."""
        self.state = (self.state * self.state) % self.n
        return self.state % 2

    def next_byte(self):
        """Generate the next byte (8 bits) in the sequence."""
        byte = 0
        for i in range(8):
            byte = (byte << 1) | self.next_bit()
        return byte

    def generate_bits(self, num_bits):
        """Generate a specified number of random bits.
        
        Args:
            num_bits (int): Number of bits to generate
            
        Returns:
            list: List of generated bits (0s and 1s)
        """
        return [self.next_bit() for _ in range(num_bits)]

    def generate_bytes(self, num_bytes):
        """Generate a specified number of random bytes.
        
        Args:
            num_bytes (int): Number of bytes to generate
            
        Returns:
            bytes: Generated random bytes
        """
        return bytes(self.next_byte() for _ in range(num_bytes))

In [5]:
p = 11 
q = 19 
seed = 3 
bbs = BBS(p, q, seed) 
print(bbs.generate_bits(10)) 

[1, 0, 0, 0, 0, 0, 1, 0, 1, 1]


In [ ]:
class LCG:
    def __init__(self, seed, a, c, m):
        """Initialize Linear Congruential Generator.
        
        Args:
            seed (int): Starting value
            a (int): Multiplier
            c (int): Increment
            m (int): Modulus
            
        The sequence is generated using the formula:
        X_(n+1) = (a * X_n + c) mod m
        """
        self.state = seed
        self.a = a
        self.c = c
        self.m = m
        
        # Validate parameters
        if m <= 0:
            raise ValueError("Modulus m must be positive")
        if a <= 0:
            raise ValueError("Multiplier a must be positive")
        if c < 0:
            raise ValueError("Increment c must be non-negative")
        if seed < 0:
            raise ValueError("Seed must be non-negative")

    def next(self):
        """Generate the next number in the sequence."""
        self.state = (self.a * self.state + self.c) % self.m
        return self.state

    def next_float(self):
        """Generate the next number as a float between 0 and 1."""
        return self.next() / self.m

    def generate_sequence(self, n):
        """Generate a sequence of n random numbers.
        
        Args:
            n (int): Number of values to generate
            
        Returns:
            list: List of generated numbers
        """
        return [self.next() for _ in range(n)]

class StandardLCG:
    """Predefined LCG parameters for common implementations."""
    
    @staticmethod
    def java_random(seed):
        """Create LCG with Java's Random parameters.
        multiplier = 0x5DEECE66D
        addend = 0xB
        mask = (1 << 48) - 1
        """
        return LCG(seed, 0x5DEECE66D, 0xB, (1 << 48))
    
    @staticmethod
    def minstd(seed):
        """Create LCG with MINSTD parameters.
        Used in older versions of the C++ standard library.
        """
        return LCG(seed, 16807, 0, (1 << 31) - 1)
    
    @staticmethod
    def glibc(seed):
        """Create LCG with parameters used in glibc rand()."""
        return LCG(seed, 1103515245, 12345, (1 << 31))

    @staticmethod
    def numerical_recipes(seed):
        """Create LCG with parameters from Numerical Recipes."""
        return LCG(seed, 1664525, 1013904223, (1 << 32))

In [ ]:
class SimplePRNG:
    def __init__(self, seed):
        """Initialize a simple random number generator.
        
        Args:
            seed: Starting value for the generator
        """
        self.state = seed
        
        # Using simple common parameters
        self.a = 1664525
        self.c = 1013904223
        self.m = 2**32
    
    def next_number(self):
        """Generate next random number."""
        self.state = (self.a * self.state + self.c) % self.m
        return self.state
    
    def random_range(self, min_val, max_val):
        """Generate random number between min_val and max_val."""
        num = self.next_number()
        # Scale the number to the desired range
        return min_val + (num % (max_val - min_val + 1))

# Example usage:
if __name__ == "__main__":
    # Create generator with seed
    rng = SimplePRNG(12345)
    
    # Generate 5 random numbers
    print("Random numbers:")
    for _ in range(5):
        print(rng.next_number())
    
    # Generate 5 random numbers between 1 and 100
    print("\nRandom numbers between 1 and 100:")
    for _ in range(5):
        print(rng.random_range(1, 100))

In [6]:
class SimpleRandomGenerator:
    def __init__(self, seed, multiplier, increment, modulus):
        """
        A simple random number generator using the linear congruential method.
        Formula: next_number = (multiplier * current + increment) % modulus
        
        Args:
            seed: Starting number
            multiplier: Number to multiply by
            increment: Number to add
            modulus: Number to divide by (using remainder)
        """
        self.current = seed
        self.multiplier = multiplier
        self.increment = increment
        self.modulus = modulus
    
    def next_number(self):
        """Generate the next random number using the formula."""
        self.current = (self.multiplier * self.current + self.increment) % self.modulus
        return self.current
    
    def next_in_range(self, min_value, max_value):
        """Get a random number between min_value and max_value (inclusive)."""
        number = self.next_number()
        range_size = max_value - min_value + 1
        return min_value + (number % range_size)

# Example of how to use it:
def example_usage():
    # Create a random generator
    # Using small numbers for the example
    random = SimpleRandomGenerator(
        seed=5,         # Starting number
        multiplier=7,   # Number to multiply by
        increment=3,    # Number to add
        modulus=100     # Take remainder when dividing by this
    )
    
    # Generate some random numbers
    print("Basic random numbers:")
    for _ in range(5):
        print(random.next_number())
    
    # Generate numbers in a specific range (like rolling dice)
    print("\nNumbers between 1 and 6 (like a dice):")
    for _ in range(5):
        print(random.next_in_range(1, 6))

if __name__ == "__main__":
    example_usage()

Basic random numbers:
38
69
86
5
38

Numbers between 1 and 6 (like a dice):
4
3
6
3
4


In [ ]:
class SimpleRandomGenerator:
    def __init__(self, seed, multiplier, increment, modulus):
        """
        A simple random number generator using the linear congruential method.
        Formula: next_number = (multiplier * current + increment) % modulus
        
        Args:
            seed: Starting number
            multiplier: Number to multiply by
            increment: Number to add
            modulus: Number to divide by (using remainder)
        """
        self.current = seed
        self.multiplier = multiplier
        self.increment = increment
        self.modulus = modulus
    
    def next_number(self):
        """Generate the next random number using the formula."""
        self.current = (self.multiplier * self.current + self.increment) % self.modulus
        return self.current
    
    def next_in_range(self, min_value, max_value):
        """Get a random number between min_value and max_value (inclusive)."""
        number = self.next_number()
        range_size = max_value - min_value + 1
        return min_value + (number % range_size)

# Example of how to use it:
def example_usage():
    # Create a random generator
    # Using small numbers for the example
    random = SimpleRandomGenerator(
        seed=5,         # Starting number
        multiplier=7,   # Number to multiply by
        increment=3,    # Number to add
        modulus=100     # Take remainder when dividing by this
    )
    
    # Generate some random numbers
    print("Basic random numbers:")
    for _ in range(5):
        print(random.next_number())
    
    # Generate numbers in a specific range (like rolling dice)
    print("\nNumbers between 1 and 6 (like a dice):")
    for _ in range(5):
        print(random.next_in_range(1, 6))

if __name__ == "__main__":
    example_usage()

In [ ]:
def periodicity_test(generator, max_iterations=1000000):
    """
    Test the periodicity of a PRNG by detecting cycles in the sequence.
    
    Parameters:
    generator: function or iterator
        A function that generates the next random number in the sequence
    max_iterations: int
        Maximum number of iterations to test before giving up
        
    Returns:
    tuple: (period_length, sequence)
        period_length: Length of the detected cycle (0 if no cycle found)
        sequence: List of values in the detected cycle
    """
    seen_values = {}
    sequence = []
    
    for i in range(max_iterations):
        try:
            # Get next value from generator
            current = next(generator) if hasattr(generator, '__next__') else generator()
            
            # Store value and its position
            if current in seen_values:
                # Cycle detected
                cycle_start = seen_values[current]
                cycle_length = i - cycle_start
                cycle_sequence = sequence[cycle_start:i]
                return cycle_length, cycle_sequence
            
            seen_values[current] = i
            sequence.append(current)
            
        except StopIteration:
            # Generator exhausted
            return 0, sequence
    
    # No cycle found within max_iterations
    return 0, sequence

# Example usage with a simple Linear Congruential Generator (LCG)
def create_lcg(modulus, multiplier, increment, seed):
    """
    Create a Linear Congruential Generator
    X_(n+1) = (multiplier * X_n + increment) mod modulus
    """
    def lcg():
        nonlocal seed
        seed = (multiplier * seed + increment) % modulus
        return seed
    return lcg

# Helper function to demonstrate the test
def test_prng(generator, name="PRNG"):
    """
    Run periodicity test and print results
    """
    period, cycle = periodicity_test(generator)
    
    print(f"\nTesting {name}:")
    if period > 0:
        print(f"Period length: {period}")
        print(f"Cycle values: {cycle[:min(10, len(cycle))]}{'...' if len(cycle) > 10 else ''}")
    else:
        print("No cycle detected within maximum iterations")
        if len(cycle) > 0:
            print(f"Generated sequence (first 10): {cycle[:10]}...")

# Example demonstration
if __name__ == "__main__":
    # Test a bad LCG with small parameters
    bad_lcg = create_lcg(modulus=16, multiplier=5, increment=3, seed=7)
    test_prng(bad_lcg, "Bad LCG (small parameters)")
    
    # Test a better LCG with larger parameters
    better_lcg = create_lcg(modulus=2**31 - 1, multiplier=1103515245, increment=12345, seed=42)
    test_prng(better_lcg, "Better LCG (larger parameters)")

In [ ]:
import math
from collections import Counter
import numpy as np

def periodicity_test(generator, max_iterations=1000000):
    """
    Test the periodicity of a PRNG by detecting cycles in the sequence.
    
    Parameters:
    generator: function or iterator
        A function that generates the next random number in the sequence
    max_iterations: int
        Maximum number of iterations to test before giving up
        
    Returns:
    tuple: (period_length, sequence)
        period_length: Length of the detected cycle (0 if no cycle found)
        sequence: List of values in the detected cycle
    """
    seen_values = {}
    sequence = []
    
    for i in range(max_iterations):
        try:
            current = next(generator) if hasattr(generator, '__next__') else generator()
            if current in seen_values:
                cycle_start = seen_values[current]
                cycle_length = i - cycle_start
                cycle_sequence = sequence[cycle_start:i]
                return cycle_length, cycle_sequence
            
            seen_values[current] = i
            sequence.append(current)
            
        except StopIteration:
            return 0, sequence
    
    return 0, sequence

def entropy_test(generator, num_samples=10000, num_bins=256):
    """
    Calculate the entropy of a PRNG output sequence.
    
    Parameters:
    generator: function or iterator
        A function that generates the next random number in the sequence
    num_samples: int
        Number of samples to generate
    num_bins: int
        Number of bins for discretizing continuous values
        
    Returns:
    dict: Dictionary containing various entropy metrics
    """
    # Generate sequence
    sequence = []
    for _ in range(num_samples):
        try:
            value = next(generator) if hasattr(generator, '__next__') else generator()
            sequence.append(value)
        except StopIteration:
            break
    
    # Convert sequence to numpy array
    sequence = np.array(sequence)
    
    # Normalize and discretize values if they're not already discrete
    if not all(isinstance(x, int) for x in sequence):
        sequence = np.floor((sequence - sequence.min()) / 
                          (sequence.max() - sequence.min()) * (num_bins - 1)).astype(int)
    
    # Calculate probabilities
    counts = Counter(sequence)
    total = len(sequence)
    probabilities = [count/total for count in counts.values()]
    
    # Calculate Shannon entropy
    shannon_entropy = -sum(p * math.log2(p) for p in probabilities)
    max_entropy = math.log2(len(counts))
    normalized_entropy = shannon_entropy / max_entropy if max_entropy > 0 else 0
    
    # Calculate additional entropy metrics
    collision_entropy = -math.log2(sum(p*p for p in probabilities))
    min_entropy = -math.log2(max(probabilities))
    
    # Calculate chi-square statistic for uniformity test
    expected_freq = total / len(counts)
    chi_square = sum((count - expected_freq)**2 / expected_freq 
                    for count in counts.values())
    
    return {
        'shannon_entropy': shannon_entropy,
        'normalized_entropy': normalized_entropy,
        'collision_entropy': collision_entropy,
        'min_entropy': min_entropy,
        'chi_square': chi_square,
        'num_unique_values': len(counts),
        'sample_size': total
    }

def create_lcg(modulus, multiplier, increment, seed):
    """
    Create a Linear Congruential Generator
    X_(n+1) = (multiplier * X_n + increment) mod modulus
    """
    def lcg():
        nonlocal seed
        seed = (multiplier * seed + increment) % modulus
        return seed / modulus  # Normalize to [0,1)
    return lcg

def test_prng(generator, name="PRNG"):
    """
    Run comprehensive PRNG tests and print results
    """
    print(f"\nTesting {name}:")
    
    # Periodicity test
    period, cycle = periodicity_test(generator)
    print("\nPeriodicity Test Results:")
    if period > 0:
        print(f"Period length: {period}")
        print(f"Cycle values: {cycle[:min(10, len(cycle))]}{'...' if len(cycle) > 10 else ''}")
    else:
        print("No cycle detected within maximum iterations")
    
    # Reset generator for entropy test
    if hasattr(generator, '__code__'):  # If it's a function
        generator = create_lcg(generator.__closure__[0].cell_contents,  # modulus
                             generator.__closure__[1].cell_contents,    # multiplier
                             generator.__closure__[2].cell_contents,    # increment
                             generator.__closure__[3].cell_contents)    # seed
    
    # Entropy test
    entropy_results = entropy_test(generator)
    print("\nEntropy Test Results:")
    print(f"Shannon Entropy: {entropy_results['shannon_entropy']:.4f} bits")
    print(f"Normalized Entropy: {entropy_results['normalized_entropy']:.4f}")
    print(f"Collision Entropy: {entropy_results['collision_entropy']:.4f} bits")
    print(f"Min Entropy: {entropy_results['min_entropy']:.4f} bits")
    print(f"Chi-square statistic: {entropy_results['chi_square']:.4f}")
    print(f"Number of unique values: {entropy_results['num_unique_values']}")
    print(f"Sample size: {entropy_results['sample_size']}")

# Example usage
if __name__ == "__main__":
    # Test a poor PRNG (small parameters)
    bad_lcg = create_lcg(modulus=16, multiplier=5, increment=3, seed=7)
    test_prng(bad_lcg, "Bad LCG (small parameters)")
    
    # Test a better PRNG (larger parameters)
    better_lcg = create_lcg(modulus=2**31 - 1, multiplier=1103515245, increment=12345, seed=42)
    test_prng(better_lcg, "Better LCG (larger parameters)")